In [1]:
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = r"/home/feliciano/LOBSTER SOUNDS/DatasetClass/"
SAMPLE_RATE = 8000

# Frequency bands
BASEBAND_BAND = (0, 200)
BUZZ_BAND = (350, 550)
TRANSIENT_BAND = (600, 1200)

# Detection threshold
ENERGY_THRESHOLD = 0.5

# Output folders
OUTPUT_DIR = "lobster_spectrograms"
GRID_DIR = "lobster_grids"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GRID_DIR, exist_ok=True)


# ---------------------------------------------------------
# BAND ENERGY
# ---------------------------------------------------------
def band_energy(y, sr, fmin, fmax):
    S = np.abs(librosa.stft(y, n_fft=1024))
    freqs = librosa.fft_frequencies(sr=sr)
    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    if len(idx) == 0:
        return 0.0
    return np.mean(S[idx, :])


# ---------------------------------------------------------
# DETECT EVENTS
# ---------------------------------------------------------
def detect_events(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    baseband = band_energy(y, sr, *BASEBAND_BAND)
    buzz = band_energy(y, sr, *BUZZ_BAND)
    transient = band_energy(y, sr, *TRANSIENT_BAND)

    return {
        "baseband": baseband > ENERGY_THRESHOLD,
        "buzz": buzz > ENERGY_THRESHOLD,
        "transient": transient > ENERGY_THRESHOLD,
        "energies": (baseband, buzz, transient)
    }


# ---------------------------------------------------------
# PLOT SPECTROGRAM WITH EVENT OVERLAY
# ---------------------------------------------------------
def save_spectrogram(file_path, class_name, index):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    S = librosa.stft(y, n_fft=1024)
    S_db = librosa.amplitude_to_db(abs(S), ref=np.max)

    plt.figure(figsize=(10, 6))
    librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="hz", cmap="magma")

    # Overlay event bands
    plt.axhspan(*BASEBAND_BAND, color="cyan", alpha=0.2)
    plt.axhspan(*BUZZ_BAND, color="lime", alpha=0.2)
    plt.axhspan(*TRANSIENT_BAND, color="red", alpha=0.2)

    plt.title(f"{class_name} — {os.path.basename(file_path)}")
    plt.colorbar(format="%+2.0f dB")

    out_path = f"{OUTPUT_DIR}/{class_name}_{index}.png"
    plt.savefig(out_path, dpi=150)
    plt.close()

    return out_path


# ---------------------------------------------------------
# MAIN PROCESS: SELECT 2 FILES PER CLASS
# ---------------------------------------------------------
selected = {}

for root, dirs, files in os.walk(DATASET_PATH):
    class_name = os.path.basename(root)

    for file in files:
        if not file.lower().endswith(".wav"):
            continue

        file_path = os.path.join(root, file)
        result = detect_events(file_path)

        # Only keep files with YES/YES/YES
        if result["baseband"] and result["buzz"] and result["transient"]:
            if class_name not in selected:
                selected[class_name] = []

            if len(selected[class_name]) < 2:
                selected[class_name].append(file_path)


# ---------------------------------------------------------
# GENERATE SPECTROGRAMS + GRIDS
# ---------------------------------------------------------
for class_name, file_list in selected.items():
    print(f"\nProcessing class: {class_name}")

    saved_paths = []
    for i, file_path in enumerate(file_list):
        saved = save_spectrogram(file_path, class_name, i)
        saved_paths.append(saved)

    # Create grid
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, img_path in zip(axes, saved_paths):
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.axis("off")

    plt.suptitle(f"Spectrogram Grid — {class_name}", fontsize=18)
    plt.tight_layout()

    grid_path = f"{GRID_DIR}/{class_name}_grid.png"
    plt.savefig(grid_path, dpi=150)
    plt.close()

print("\nDone. Spectrograms and grids saved.")



Processing class: juvenile_lobsters

Processing class: adult_lobsters

Processing class: female_lobsters

Processing class: male_lobsters

Done. Spectrograms and grids saved.


In [2]:
import os
import shutil
import librosa
import numpy as np

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = r"/home/feliciano/LOBSTER SOUNDS/DatasetClass/"
OUTPUT_PATH = "lobster_audio_extracts"
SAMPLE_RATE = 8000

# Frequency bands
BASEBAND_BAND = (0, 200)
BUZZ_BAND = (350, 550)
TRANSIENT_BAND = (600, 1200)

ENERGY_THRESHOLD = 0.5
N_PER_CLASS = 4

os.makedirs(OUTPUT_PATH, exist_ok=True)


# ---------------------------------------------------------
# BAND ENERGY
# ---------------------------------------------------------
def band_energy(y, sr, fmin, fmax):
    S = np.abs(librosa.stft(y, n_fft=1024))
    freqs = librosa.fft_frequencies(sr=sr)
    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    if len(idx) == 0:
        return 0.0
    return np.mean(S[idx, :])


# ---------------------------------------------------------
# DETECT EVENTS
# ---------------------------------------------------------
def detect_events(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    baseband = band_energy(y, sr, *BASEBAND_BAND)
    buzz = band_energy(y, sr, *BUZZ_BAND)
    transient = band_energy(y, sr, *TRANSIENT_BAND)

    return (
        baseband > ENERGY_THRESHOLD,
        buzz > ENERGY_THRESHOLD,
        transient > ENERGY_THRESHOLD,
        (baseband, buzz, transient)
    )


# ---------------------------------------------------------
# MAIN EXTRACTION LOOP
# ---------------------------------------------------------
selected = {}

for root, dirs, files in os.walk(DATASET_PATH):
    class_name = os.path.basename(root)

    for file in files:
        if not file.lower().endswith(".wav"):
            continue

        file_path = os.path.join(root, file)

        baseband_ok, buzz_ok, transient_ok, energies = detect_events(file_path)

        # Only keep YES/YES/YES
        if baseband_ok and buzz_ok and transient_ok:
            if class_name not in selected:
                selected[class_name] = []

            if len(selected[class_name]) < N_PER_CLASS:
                selected[class_name].append((file_path, energies))


# ---------------------------------------------------------
# COPY AUDIO FILES TO OUTPUT FOLDER
# ---------------------------------------------------------
for class_name, file_list in selected.items():
    class_dir = os.path.join(OUTPUT_PATH, class_name)
    os.makedirs(class_dir, exist_ok=True)

    print(f"\nExtracting audio for class: {class_name}")

    for i, (file_path, energies) in enumerate(file_list):
        out_path = os.path.join(class_dir, f"{class_name}_{i}.wav")
        shutil.copy(file_path, out_path)

        print(f"Saved: {out_path}")
        print(f"Energies: {energies}")



Extracting audio for class: juvenile_lobsters
Saved: lobster_audio_extracts/juvenile_lobsters/juvenile_lobsters_0.wav
Energies: (13.479778, 3.2706203, 3.1364129)
Saved: lobster_audio_extracts/juvenile_lobsters/juvenile_lobsters_1.wav
Energies: (23.516632, 3.2874959, 3.447913)
Saved: lobster_audio_extracts/juvenile_lobsters/juvenile_lobsters_2.wav
Energies: (7.7716413, 3.101068, 2.3921301)
Saved: lobster_audio_extracts/juvenile_lobsters/juvenile_lobsters_3.wav
Energies: (10.764649, 4.2115417, 2.2922342)

Extracting audio for class: adult_lobsters
Saved: lobster_audio_extracts/adult_lobsters/adult_lobsters_0.wav
Energies: (21.851614, 6.497716, 8.176305)
Saved: lobster_audio_extracts/adult_lobsters/adult_lobsters_1.wav
Energies: (2.2298477, 1.0183092, 0.8718169)
Saved: lobster_audio_extracts/adult_lobsters/adult_lobsters_2.wav
Energies: (12.498568, 1.784209, 1.4417675)
Saved: lobster_audio_extracts/adult_lobsters/adult_lobsters_3.wav
Energies: (2.6511967, 0.8908493, 0.964763)

Extracting

In [5]:
import os
import librosa
import numpy as np
import soundfile as sf
from scipy.signal import butter, filtfilt

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = r"/home/feliciano/LOBSTER SOUNDS/LobsterSounds_5s"
OUTPUT_PATH = "lobster_rasp_extracts"
SAMPLE_RATE = 8000

# Antennal rasp band (broadband)
LOW_CUT = 300
HIGH_CUT = 1500

# Extraction window around detected rasp (ms)
PRE_MS = 30
POST_MS = 80

# Number of pulses per class
N_PER_CLASS = 4

os.makedirs(OUTPUT_PATH, exist_ok=True)


# ---------------------------------------------------------
# BANDPASS FILTER (broadband rasp)
# ---------------------------------------------------------
def bandpass_filter(y, sr, low, high):
    nyq = sr / 2
    b, a = butter(4, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, y)


# ---------------------------------------------------------
# EXTRACT RASP PULSES (biological signature)
# ---------------------------------------------------------
def extract_rasp_pulses(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    # 1. Band-pass filter to isolate rasping frequencies
    y_filt = bandpass_filter(y, sr, LOW_CUT, HIGH_CUT)

    # 2. Spectral flux (excellent for scratch/rasp detection)
    flux = librosa.onset.onset_strength(y=y_filt, sr=sr)

    # 3. Adaptive threshold
    threshold = np.mean(flux) + 2.5 * np.std(flux)

    # 4. Detect peaks above threshold
    peaks = np.where(flux > threshold)[0]
    if len(peaks) == 0:
        return []

    # 5. Group peaks into pulses (rasps often have multiple sub-peaks)
    grouped = []
    current_group = [peaks[0]]

    for p in peaks[1:]:
        if p - current_group[-1] <= 3:  # within ~40 ms
            current_group.append(p)
        else:
            grouped.append(current_group)
            current_group = [p]
    grouped.append(current_group)

    # 6. Extract audio around each grouped rasp
    pulses = []
    pre = int((PRE_MS / 1000) * sr)
    post = int((POST_MS / 1000) * sr)

    for group in grouped:
        center = int(np.mean(group))
        start = max(0, center - pre)
        end = min(len(y), center + post)
        pulses.append(y[start:end])

    return pulses


# ---------------------------------------------------------
# MAIN LOOP: EXTRACT 4 RASPS PER CLASS
# ---------------------------------------------------------
for root, dirs, files in os.walk(DATASET_PATH):
    class_name = os.path.basename(root)
    class_out = os.path.join(OUTPUT_PATH, class_name)
    os.makedirs(class_out, exist_ok=True)

    saved = 0

    for file in files:
        if not file.lower().endswith(".wav"):
            continue

        file_path = os.path.join(root, file)
        pulses = extract_rasp_pulses(file_path)

        for pulse in pulses:
            if saved >= N_PER_CLASS:
                break

            out_file = os.path.join(class_out, f"{class_name}_rasp_{saved}.wav")
            sf.write(out_file, pulse, SAMPLE_RATE)
            saved += 1

        if saved >= N_PER_CLASS:
            break

    print(f"Extracted {saved} rasp pulses for class: {class_name}")


Extracted 0 rasp pulses for class: LobsterSounds_5s
Extracted 4 rasp pulses for class: male_lobsters5
Extracted 4 rasp pulses for class: female_lobsters5
Extracted 4 rasp pulses for class: juvenile_lobsters5
Extracted 4 rasp pulses for class: adult_lobsters5


In [8]:
import os
import librosa
import numpy as np
import soundfile as sf
from sklearn.cluster import KMeans

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = r"/home/feliciano/LOBSTER SOUNDS/LobsterSounds_5s"
OUTPUT_PATH = "lobster_clusters"
SAMPLE_RATE = 8000

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Frequency bands (adapted for 8 kHz)
BANDS = [
    (0, 200),     # baseband / rumble
    (200, 600),   # low-mid (buzz/rasp)
    (600, 1500),  # mid-high (clicks/rasps)
    (1500, 3500)  # high (sharp clicks)
]

N_FFT = 128   # small FFT so short segments work
HOP = 64


# ---------------------------------------------------------
# SAFE BAND ENERGY
# ---------------------------------------------------------
def band_energy(y, sr, fmin, fmax):
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

    # Clip band to valid frequency range
    fmin = max(fmin, freqs[0])
    fmax = min(fmax, freqs[-1])

    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]

    if len(idx) == 0:
        return 0.0

    # Clip index to avoid out-of-bounds
    idx = idx[idx < S.shape[0]]

    if len(idx) == 0:
        return 0.0

    return float(np.mean(S[idx, :]))


# ---------------------------------------------------------
# EXTRACT EVENTS (SHORT SOUND SEGMENTS)
# ---------------------------------------------------------
def extract_events(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    # Onset detection
    onset_frames = librosa.onset.onset_detect(y=y, sr=sr, backtrack=True)
    events = []

    # Window around each onset (80 ms)
    win_ms = 80
    win = int((win_ms / 1000) * sr)

    for f in onset_frames:
        center = librosa.frames_to_samples(f)
        start = max(0, center - win // 2)
        end = min(len(y), center + win // 2)
        seg = y[start:end]

        # Skip too-short segments (<20 ms)
        if len(seg) < int(0.02 * sr):
            continue

        events.append(seg)

    return events


# ---------------------------------------------------------
# FEATURE EXTRACTION
# ---------------------------------------------------------
def event_features(seg, sr):
    feats = []

    # Duration
    feats.append(len(seg) / sr)

    # RMS
    feats.append(float(np.sqrt(np.mean(seg**2))))

    # Band energies
    for (fmin, fmax) in BANDS:
        feats.append(band_energy(seg, sr, fmin, fmax))

    # Spectral centroid & bandwidth
    S = np.abs(librosa.stft(seg, n_fft=N_FFT, hop_length=HOP))
    if S.size == 0:
        feats += [0.0, 0.0]
    else:
        centroid = librosa.feature.spectral_centroid(S=S, sr=sr).mean()
        bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr).mean()
        feats += [float(centroid), float(bandwidth)]

    return np.array(feats, dtype=float)


# ---------------------------------------------------------
# COLLECT EVENTS + FEATURES
# ---------------------------------------------------------
all_features = []
all_segments = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if not file.lower().endswith(".wav"):
            continue

        file_path = os.path.join(root, file)
        events = extract_events(file_path)

        for seg in events:
            feats = event_features(seg, SAMPLE_RATE)
            all_features.append(feats)
            all_segments.append(seg)

print(f"Total events collected: {len(all_segments)}")

all_features = np.vstack(all_features)


# ---------------------------------------------------------
# CLUSTER EVENTS
# ---------------------------------------------------------
n_clusters = 6
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(all_features)

print("Cluster counts:", np.bincount(labels))


# ---------------------------------------------------------
# SAVE EXAMPLE AUDIO PER CLUSTER
# ---------------------------------------------------------
examples_per_cluster = 10

for c in range(n_clusters):
    cluster_dir = os.path.join(OUTPUT_PATH, f"cluster_{c}")
    os.makedirs(cluster_dir, exist_ok=True)

    idxs = np.where(labels == c)[0][:examples_per_cluster]

    for i, idx in enumerate(idxs):
        seg = all_segments[idx]
        out_path = os.path.join(cluster_dir, f"cluster_{c}_example_{i}.wav")
        sf.write(out_path, seg, SAMPLE_RATE)

    print(f"Saved {len(idxs)} examples for cluster {c}")


Total events collected: 46664
Cluster counts: [11307  7973  4009 12913  2056  8406]
Saved 10 examples for cluster 0
Saved 10 examples for cluster 1
Saved 10 examples for cluster 2
Saved 10 examples for cluster 3
Saved 10 examples for cluster 4
Saved 10 examples for cluster 5


In [10]:
import os
import librosa
import numpy as np
import soundfile as sf
from sklearn.cluster import MiniBatchKMeans
from scipy.stats import kurtosis, skew

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = r"/home/feliciano/LOBSTER SOUNDS/LobsterSounds_5s"
OUTPUT_PATH = "lobster_clusters"
SAMPLE_RATE = 8000

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Biological frequency bands
BANDS = [
    (0, 200),
    (200, 600),
    (600, 1500),
    (1500, 3500)
]

N_FFT = 128
HOP = 64


# ---------------------------------------------------------
# SAFE BAND ENERGY
# ---------------------------------------------------------
def band_energy(y, sr, fmin, fmax):
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))
    if S.size == 0:
        return 0.0

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

    fmin = max(fmin, freqs[0])
    fmax = min(fmax, freqs[-1])

    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    idx = idx[idx < S.shape[0]]

    if len(idx) == 0:
        return 0.0

    return float(np.mean(S[idx, :]))


# ---------------------------------------------------------
# BIOLOGICAL EVENT DETECTION
# ---------------------------------------------------------
def extract_events(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    flux = librosa.onset.onset_strength(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    flat = librosa.feature.spectral_flatness(y=y)[0]

    flux_thr = np.mean(flux) + 2 * np.std(flux)
    zcr_thr = np.mean(zcr) + 1.5 * np.std(zcr)
    flat_thr = np.mean(flat) + 1.5 * np.std(flat)

    frames = np.where(
        (flux > flux_thr) |
        (zcr > zcr_thr) |
        (flat > flat_thr)
    )[0]

    events = []
    win = int(0.08 * sr)

    for f in frames:
        center = librosa.frames_to_samples(f)
        start = max(0, center - win // 2)
        end = min(len(y), center + win // 2)
        seg = y[start:end]

        if len(seg) < int(0.02 * sr):
            continue

        events.append(seg)

    return events


# ---------------------------------------------------------
# FEATURE EXTRACTION (ROBUST)
# ---------------------------------------------------------
def event_features(seg, sr):
    feats = []

    feats.append(len(seg) / sr)                     # duration
    feats.append(float(np.sqrt(np.mean(seg**2))))   # RMS
    feats.append(float(np.mean(np.abs(seg))))       # amplitude
    feats.append(float(np.mean(np.diff(seg)**2)))   # roughness

    for (fmin, fmax) in BANDS:
        feats.append(band_energy(seg, sr, fmin, fmax))

    S = np.abs(librosa.stft(seg, n_fft=N_FFT, hop_length=HOP))
    if S.size == 0:
        feats += [0, 0, 0]
    else:
        feats.append(float(librosa.feature.spectral_centroid(S=S, sr=sr).mean()))
        feats.append(float(librosa.feature.spectral_bandwidth(S=S, sr=sr).mean()))
        feats.append(float(librosa.feature.spectral_flatness(S=S).mean()))

    feats.append(float(kurtosis(seg, nan_policy='omit')))
    feats.append(float(skew(seg, nan_policy='omit')))

    # Replace NaN or inf with 0
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    return feats


# ---------------------------------------------------------
# COLLECT EVENTS + FEATURES
# ---------------------------------------------------------
all_features = []
all_segments = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if not file.lower().endswith(".wav"):
            continue

        file_path = os.path.join(root, file)
        events = extract_events(file_path)

        for seg in events:
            feats = event_features(seg, SAMPLE_RATE)
            all_features.append(feats)
            all_segments.append(seg)

print(f"Total events collected: {len(all_segments)}")

all_features = np.vstack(all_features)


# ---------------------------------------------------------
# CLUSTER EVENTS (ROBUST)
# ---------------------------------------------------------
n_clusters = 6
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=2048)
labels = kmeans.fit_predict(all_features)

print("Cluster counts:", np.bincount(labels))


# ---------------------------------------------------------
# SAVE EXAMPLE AUDIO PER CLUSTER
# ---------------------------------------------------------
examples_per_cluster = 10

for c in range(n_clusters):
    cluster_dir = os.path.join(OUTPUT_PATH, f"cluster_{c}")
    os.makedirs(cluster_dir, exist_ok=True)

    idxs = np.where(labels == c)[0][:examples_per_cluster]

    for i, idx in enumerate(idxs):
        seg = all_segments[idx]
        out_path = os.path.join(cluster_dir, f"cluster_{c}_example_{i}.wav")
        sf.write(out_path, seg, SAMPLE_RATE)

    print(f"Saved {len(idxs)} examples for cluster {c}")


Total events collected: 38074
Cluster counts: [ 6136  6325 10362  1735 10881  2635]
Saved 10 examples for cluster 0
Saved 10 examples for cluster 1
Saved 10 examples for cluster 2
Saved 10 examples for cluster 3
Saved 10 examples for cluster 4
Saved 10 examples for cluster 5


In [2]:
import os
import librosa
import numpy as np
import soundfile as sf
from sklearn.cluster import MiniBatchKMeans

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = "/home/feliciano/LOBSTER SOUNDS/LobsterSounds_5s"
OUTPUT_PATH = "lobster_full_segment_clusters"
SAMPLE_RATE = 8000

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_FFT = 512
HOP = 256

# ---------------------------------------------------------
# FEATURE EXTRACTION FOR FULL 5-SECOND SEGMENT
# ---------------------------------------------------------
def segment_features(y, sr):
    feats = []

    # Duration (should be ~5 sec)
    feats.append(len(y) / sr)

    # RMS energy
    feats.append(float(np.sqrt(np.mean(y**2))))

    # Mean absolute amplitude
    feats.append(float(np.mean(np.abs(y))))

    # Roughness (derivative energy)
    feats.append(float(np.mean(np.diff(y)**2)))

    # Spectrogram
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))

    # Spectral features
    feats.append(float(librosa.feature.spectral_centroid(S=S, sr=sr).mean()))
    feats.append(float(librosa.feature.spectral_bandwidth(S=S, sr=sr).mean()))
    feats.append(float(librosa.feature.spectral_flatness(S=S).mean()))
    feats.append(float(librosa.feature.spectral_rolloff(S=S, sr=sr).mean()))

    # Zero-crossing rate
    feats.append(float(librosa.feature.zero_crossing_rate(y).mean()))

    # Band energies
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    bands = [
        (0, 200),
        (200, 600),
        (600, 1500),
        (1500, 3500)
    ]

    for (fmin, fmax) in bands:
        idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
        idx = idx[idx < S.shape[0]]
        if len(idx) == 0:
            feats.append(0.0)
        else:
            feats.append(float(np.mean(S[idx, :])))

    # Replace NaN/inf
    return np.nan_to_num(feats)


# ---------------------------------------------------------
# COLLECT FEATURES FOR FULL SEGMENTS
# ---------------------------------------------------------
all_features = []
all_segments = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(".wav"):
            path = os.path.join(root, file)

            y, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)

            feats = segment_features(y, sr)
            all_features.append(feats)
            all_segments.append((file, y))

all_features = np.vstack(all_features)

# ---------------------------------------------------------
# CLUSTER FULL 5-SECOND SEGMENTS
# ---------------------------------------------------------
kmeans = MiniBatchKMeans(n_clusters=6, random_state=42, batch_size=512)
labels = kmeans.fit_predict(all_features)

# ---------------------------------------------------------
# SAVE EXAMPLES PER CLUSTER
# ---------------------------------------------------------
for c in range(6):
    out_dir = os.path.join(OUTPUT_PATH, f"cluster_{c}")
    os.makedirs(out_dir, exist_ok=True)

    idxs = np.where(labels == c)[0][:10]

    for i, idx in enumerate(idxs):
        fname, audio = all_segments[idx]
        sf.write(os.path.join(out_dir, f"{i}_{fname}"), audio, SAMPLE_RATE)
